# 01 — Obtenção e validação

Verificar se a versão 2 de `akashkr/phishing-website-dataset` corresponde ao PhiUSIIL. Este notebook lê a auditoria persistida pelo comando `python scripts/download_data.py`. A inspeção compara esquema, dimensão, alvo e integridade dos arquivos, sempre em fontes separadas.

In [1]:
import json
from pathlib import Path

cwd = Path.cwd()
root = next(path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file())
record = root / "results" / "data_audit" / "eedb92afaef04c27b08941e64df07dfc.json"
audit = json.loads(record.read_text(encoding="utf-8"))
assert not audit["errors"]
kaggle = audit["kaggle"]["tables"][0]
uci = audit["uci"]

print(f"{'Fonte':<14}{'Registros':>12}{'Colunas':>10}  Alvo")
print(f"{'Kaggle v2':<14}{kaggle['rows']:>12}{kaggle['column_count']:>10}  {kaggle['target']}")
print(f"{'UCI 967':<14}{uci['rows']:>12}{uci['column_count']:>10}  {uci['target']}")
print("Classes originais UCI:", uci["target_counts"])
print("Codificação interna:", uci["internal_encoding"])
print("SHA-256 do CSV UCI:", uci["csv_sha256"])

Fonte            Registros   Colunas  Alvo
Kaggle v2            11055        32  Result
UCI 967             235795        56  label
Classes originais UCI: {'1': 134850, '0': 100945}
Codificação interna: {'0': 1, '1': 0}
SHA-256 do CSV UCI: a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6


## Interpretação e conclusão

O arquivo da Kaggle contém 32 colunas, usa `Result` como variável-alvo e tem menos de 5% da quantidade de registros da UCI. O CSV da UCI confirma os 54 preditores catalogados, o identificador e o alvo `label`. A análise usará a fonte UCI. O passo seguinte é auditar duplicatas, conflitos de rótulo e informações de agrupamento para reservar o teste antes da exploração detalhada.